In [2]:
# load necessary modules
import numpy as np 
from scipy.integrate import odeint
import os, sys 
from pathlib import Path
from os.path import dirname, realpath
script_dir = Path(dirname(realpath('.')))
module_dir = str(script_dir)
sys.path.insert(0, module_dir + '/modules')
import utility as ut
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# L63 system
def L63(u, alpha=10., rho=28., beta=8./3.):
    x, y, z = u #np.split(u, 3, axis=-1)
    p = alpha * (y - x)
    q = (rho - z) * x - y
    r = x * y - beta * z
    return np.array([p, q, r])

# single trajectory generator for L63
def generate_trajectory(state0, dt, n_steps):
    return odeint(lambda x, t: L63(x), state0, np.arange(0, n_steps*dt, dt))

# multiple trajectories generator for L63
@ut.timer
def generate_trajectories(states0, dt, n_steps, save_folder, name):
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    trajectories = np.zeros((len(states0), 3, n_steps))
    for i, state0 in enumerate(states0):
        trajectories[i, :, :] = generate_trajectory(state0, dt, n_steps).T
    np.save('{}/{}.npy'.format(save_folder, name), trajectories)

In [16]:
seed = 43
np.random.seed(seed)
save_folder='../data/L63-trajectories'
dt = 0.02
n_steps = 200000

# find a point on the attractor
random_point =  np.random.normal(size=3)
attractor_point = generate_trajectory(random_point, 0.02, n_steps=200000)[-1]
for i in range(10):
    train = generate_trajectory(attractor_point, dt/(i+1), n_steps)
    np.save('{}/{}.npy'.format(save_folder, f'train{i+1}'), train.T)